# 🤖 AI Knowledge Base Q&A Bot

A production-ready **Retrieval-Augmented Generation (RAG)** chatbot that:
- Chunks documents intelligently with `RecursiveCharacterTextSplitter`
- Embeds with HuggingFace `all-MiniLM-L6-v2` (free, no API key)
- Stores vectors in **FAISS** for sub-millisecond retrieval
- Generates answers with **GPT-2** (runs locally)
- Deploys locally with **Streamlit**
- **100% free — no API keys required**

---

## 1. Install Dependencies

In [ ]:
%pip install "langchain<1.0" langchain-huggingface langchain-community langchain-text-splitters faiss-cpu sentence-transformers streamlit -q

## 2. Import Libraries

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

print("✅ Libraries loaded — no API key needed!")

## 3. Load Documents
Load the AI/ML knowledge base from the `data/` directory.

In [3]:
loader = DirectoryLoader("data/", glob="**/*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
documents = loader.load()

print(f"📄 Loaded {len(documents)} document(s)")
print(f"📏 Total characters: {sum(len(doc.page_content) for doc in documents):,}")

📄 Loaded 1 document(s)
📏 Total characters: 17,406


## 4. Chunk Documents
Split documents into semantically meaningful chunks with overlap for context continuity.

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
    separators=["\n=================", "\n\n", "\n", ". ", " "]
)

chunks = text_splitter.split_documents(documents)

print(f"🔪 Split into {len(chunks)} chunks")
print(f"📊 Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print(f"\n--- Sample Chunk ---")
print(chunks[0].page_content[:300])

🔪 Split into 45 chunks
📊 Average chunk size: 388 chars

--- Sample Chunk ---
ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING — COMPREHENSIVE KNOWLEDGE BASE

SECTION 1: INTRODUCTION TO ARTIFICIAL INTELLIGENCE


## 5. Create Embeddings & FAISS Vector Store
Embed all chunks using HuggingFace's `all-MiniLM-L6-v2` sentence transformer (free, local) and store in FAISS.

In [5]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Build FAISS index
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"✅ FAISS index created with {vectorstore.index.ntotal} vectors")
print(f"📐 Vector dimension: {vectorstore.index.d}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8068.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FAISS index created with 45 vectors
📐 Vector dimension: 384


## 6. Test Similarity Search
Verify the retrieval works by searching for relevant chunks.

In [6]:
query = "What is RAG and how does it work?"
results = vectorstore.similarity_search_with_score(query, k=3)

print(f"🔍 Query: {query}\n")
for i, (doc, score) in enumerate(results, 1):
    print(f"Result {i} (score: {score:.4f}):")
    print(f"{doc.page_content[:200]}...")
    print()

🔍 Query: What is RAG and how does it work?

Result 1 (score: 0.8651):
The RAG pipeline consists of the following steps:
1. Document Loading: Loading documents from various sources (PDFs, websites, databases, text files)
2. Text Splitting/Chunking: Breaking large documen...

Result 2 (score: 1.1062):

Retrieval-Augmented Generation (RAG) is an AI framework that combines the power of large language models with external...

Result 3 (score: 1.2698):
Key components in a RAG system:
- Embedding Models: OpenAI text-embedding-ada-002, text-embedding-3-small, text-embedding-3-large, HuggingFace sentence-transformers
- Vector Databases: FAISS (Facebook...



## 7. Build the QA Chain with Local LLM
Create a RetrievalQA chain using GPT-2 (runs locally, no API key) with a custom prompt.

In [8]:
# Custom prompt for professional, structured answers
prompt_template = """You are an expert AI assistant with deep knowledge of artificial intelligence, 
machine learning, and data science. Use the following context to answer the question. 
If the answer is not in the context, say "I don't have enough information to answer that."

Provide clear, well-structured answers. Use bullet points when listing items.

Context:
{context}

Question: {question}

Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Initialize local LLM (GPT-2 — free, no API key)
llm = HuggingFacePipeline.from_model_id(
    model_id="gpt2",
    task="text-generation",
    pipeline_kwargs={"max_new_tokens": 300},
)

# Build QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)

print("✅ QA Chain ready — running 100% locally!")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11378.97it/s]


✅ QA Chain ready — running 100% locally!


## 8. Ask Questions
Test the QA bot with sample questions.

In [9]:
def ask(question):
    """Ask a question and display the answer with sources."""
    result = qa_chain.invoke({"query": question})
    print(f"❓ {question}\n")
    print(f"💡 {result['result']}\n")
    print(f"📚 Sources: {len(result['source_documents'])} chunks retrieved")
    print("-" * 60)

In [10]:
ask("What is the difference between supervised and unsupervised learning?")

Both `max_new_tokens` (=300) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❓ What is the difference between supervised and unsupervised learning?

💡 You are an expert AI assistant with deep knowledge of artificial intelligence, 
machine learning, and data science. Use the following context to answer the question. 
If the answer is not in the context, say "I don't have enough information to answer that."

Provide clear, well-structured answers. Use bullet points when listing items.

Context:
2. UNSUPERVISED LEARNING: The algorithm learns from unlabeled data and tries to find hidden patterns or intrinsic structures in the input data. It is called "unsupervised" because there is no teacher or correct answers. The algorithm must figure out what is being shown on its own. Common unsupervised learning algorithms include K-Means Clustering, Hierarchical Clustering, DBSCAN, Principal Component Analysis (PCA), and Autoencoders. Unsupervised learning is used for clustering, dimensionality reduction, and anomaly detection.

3. SEMI-SUPERVISED LEARNING: This approach com

In [11]:
ask("Explain how RAG works and why it is useful.")

Both `max_new_tokens` (=300) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❓ Explain how RAG works and why it is useful.

💡 You are an expert AI assistant with deep knowledge of artificial intelligence, 
machine learning, and data science. Use the following context to answer the question. 
If the answer is not in the context, say "I don't have enough information to answer that."

Provide clear, well-structured answers. Use bullet points when listing items.

Context:
The RAG pipeline consists of the following steps:
1. Document Loading: Loading documents from various sources (PDFs, websites, databases, text files)
2. Text Splitting/Chunking: Breaking large documents into smaller, manageable chunks
3. Embedding: Converting text chunks into numerical vectors using embedding models
4. Vector Storage: Storing the embeddings in a vector database for efficient similarity search
5. Retrieval: When a user asks a question, the system finds the most relevant chunks using semantic similarity
6. Generation: The retrieved context is passed to an LLM along with the user's q

In [12]:
ask("What are the most popular vector databases and how do they compare?")

Both `max_new_tokens` (=300) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❓ What are the most popular vector databases and how do they compare?

💡 You are an expert AI assistant with deep knowledge of artificial intelligence, 
machine learning, and data science. Use the following context to answer the question. 
If the answer is not in the context, say "I don't have enough information to answer that."

Provide clear, well-structured answers. Use bullet points when listing items.

Context:
Popular vector databases include:
- FAISS (Facebook AI Similarity Search): An open-source library for efficient similarity search and clustering of dense vectors. It supports various indexing methods including flat, IVF, and HNSW indexes.
- Pinecone: A fully managed vector database designed for production use. It offers automatic scaling, real-time updates, and hybrid search capabilities.
- ChromaDB: An open-source embedding database designed for AI applications. Easy to use and integrates well with LangChain.
- Weaviate: An open-source vector search engine with built-in ma

In [13]:
ask("What is GPT-4 and how is it different from earlier models?")

Both `max_new_tokens` (=300) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❓ What is GPT-4 and how is it different from earlier models?

💡 You are an expert AI assistant with deep knowledge of artificial intelligence, 
machine learning, and data science. Use the following context to answer the question. 
If the answer is not in the context, say "I don't have enough information to answer that."

Provide clear, well-structured answers. Use bullet points when listing items.

Context:
Key LLMs include:
- GPT Series (OpenAI): GPT-1, GPT-2, GPT-3 (175 billion parameters), GPT-3.5, GPT-4, and GPT-4o. GPT stands for Generative Pre-trained Transformer. These models power ChatGPT, one of the fastest-growing consumer applications in history.
- BERT (Google): Bidirectional Encoder Representations from Transformers. Pre-trained on masked language modeling and next sentence prediction tasks. Used for understanding tasks rather than generation.
- LLaMA (Meta): Large Language Model Meta AI. An open-source family of models that has enabled widespread research and development.

## 9. Save FAISS Index to Disk
Persist the vector store so the Streamlit app can load it without re-embedding.

In [14]:
vectorstore.save_local("faiss_index")
print("💾 FAISS index saved to faiss_index/")
print("🚀 Ready for Streamlit deployment! Run: streamlit run app.py")

💾 FAISS index saved to faiss_index/
🚀 Ready for Streamlit deployment! Run: streamlit run app.py


---
## 🚀 Next Steps

Run the Streamlit chatbot interface:
```bash
streamlit run app.py
```

The app provides a beautiful chat UI powered by the same RAG pipeline built above.